# 1) About

## Purpose
This notebook transforms raw transaction data into a **temporal behavioral dataset** suitable for Machine Learning Modeling. Each row in final dataset repredents one client at one monthly snapshot, with aggregated behavioral features and a **forward lookimg** fraud label.


## Connection to EDA

In the EDA Notebook, we established:

- Dataset contains 2 AML patterns. Fan-in (multiple senders -> one receiver) and cycle (circular flows).
- Only ~0.13% of the transactions are fraudulent -> heavily imbalanced.
- Fraudulent clients demonstrate distinct behavioral signals -> 2x unique senders/ 2x received trx volume, ~60% more transactions.
- Transaction network is sparse (1.3% density), making counterparty count features discriminative.

## Outcome of the NB

- This notebook will transform raw csv files (accounts/transactions/alerts) into a client&month level dataset readt for train-test split.

## Feature Categories

1. Behavioral Intensity: Trx counts, volumes, rolling averages (7d/30d/90)
2. Fan-In SIgnals: Unique senders, sender diversity, received volume
3. Cycle Signals: Sender-receiver overlap, bidirectional behavior patterns
4. Velocity/Change: Activity spikes, dormancy-activity transitions, new counterparties.

# 2) Importd and Data Loading

In [11]:
import pandas as pd, numpy as np 
from pathlib import Path 
import matplotlib.pyplot as plt 
import seaborn as sns

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

# import raw data
data_path = Path("../data/raw")

dfs = {}
for f in data_path.glob("*.csv"):
    dfs[f.stem] = pd.read_csv(f)


# handling timestamp - assuming 2020-01-01 as the start date of the timestamp
start_date = pd.to_datetime("2020-01-01")
for i, df in dfs.items():
    if 'TIMESTAMP' in df.columns:
        df['date'] = start_date + pd.to_timedelta(df['TIMESTAMP'], unit='D')


df_accounts = dfs['accounts']
df_transactions = dfs['transactions']
df_alerts = dfs['alerts']   

# Tagging fraudulent accounts
fraudulent_accounts = list(df_accounts[df_accounts['IS_FRAUD'] == 1]['ACCOUNT_ID'].unique())


# General OW
print(f"Accounts: {len(df_accounts):,}")
print(f"Transactions: {len(df_transactions):,}")
print(f"Alerts: {len(df_alerts):,}")
print(f"Fraud accounts: {len(fraudulent_accounts)}")
print(f"Date range: {df_transactions['date'].min().date()} to {df_transactions['date'].max().date()}")


Accounts: 10,000
Transactions: 1,323,234
Alerts: 1,719
Fraud accounts: 1685
Date range: 2020-01-01 to 2020-07-18


# 3) Snapshot Calendar

Define monthly snapshot dates — each one represents a "prediction point" where we ask: 
**will this client exhibit suspicious behavior in the upcoming month?**

**Design rules:**
- Features use **only past data** (up to snapshot date)
- Labels look at the **full calendar month ahead** (snapshot month)
- Need at least **3 full calendar months of history** for rolling features
- Data range: 2020-01-01 to 2020-07-18

This means:
- **First usable snapshot:** April 1 (Jan, Feb, Mar -> 3 full months of history)
- **Last usable snapshot:** June 1 (needs full June for labels; July only has 18 days)

In [13]:
data_start = df_transactions['date'].min()
data_end = df_transactions['date'].max()
print(f"Data covers a period of {(data_end - data_start).days} days")
print(f"Date Range of Transactions: {data_start} to {data_end}")

# Create Monthly Snapshots

# min history - 3 full calendar months
min_history_months = 3

# all month starts
month_starts = pd.date_range(start=data_start, end=data_end, freq = "MS")

# snapshot_dates -> 3 calendar months of history, 1 calendar month of future

snapshot_dates = []
for i in month_starts:
    month_end = i + pd.offsets.MonthEnd(0)
    has_full_month_ahead = month_end <= data_end

    history_start = i - pd.DateOffset(months = min_history_months)
    has_enough_history = history_start >= data_start

    if has_full_month_ahead and has_enough_history:

        snapshot_dates.append(i)

print(f"\nUsable snapshot dates ({len(snapshot_dates)}):")


# print each snapshot date's rolling window history start and prediction date

for i in snapshot_dates:
    history_start = i - pd.DateOffset(months = min_history_months) 
    history_end =  i - pd.Timedelta(days=1)
    prediction_end = i + pd.offsets.MonthEnd(0)
        # print(f"SnapshotDate {i}, uses history from {history_start} to i - , {month_end}")
    print(f" {i.date()}  |  history from: {history_start.date()} to {history_end.date()} |  predicts: {i.date()} until {prediction_end.date()}")


Data covers a period of 199 days
Date Range of Transactions: 2020-01-01 00:00:00 to 2020-07-18 00:00:00

Usable snapshot dates (3):
 2020-04-01  |  history from: 2020-01-01 to 2020-03-31 |  predicts: 2020-04-01 until 2020-04-30
 2020-05-01  |  history from: 2020-02-01 to 2020-04-30 |  predicts: 2020-05-01 until 2020-05-31
 2020-06-01  |  history from: 2020-03-01 to 2020-05-31 |  predicts: 2020-06-01 until 2020-06-30


# 4) Feature Engineering

For each client at each snapshot date, we will compute behavioral futures using transactions that occured before the snapshot date.  



## 4.a. Behavioral Intensity

## 4.a Behavioral Intensity

General transaction behavior per client, computed for both **Recent** (previous 1 month) and **Baseline** (2 months before that) windows. Non-overlapping so velocity ratios give clean change signals.

Features per window:
- Transaction counts (sent/received)
- Total volume (sent/received)
- Average transaction size (sent/received)

In [ ]:
### Old Code - Calculates Features by Account - first filters the data by account then window

# def compute_intensity_features(df_trx, account_id, window_start, window_end, prefix):
#     """
#     Compute behavioral intensity features for a single account within a time window.
#     prefix: 'recent' or 'baseline' to distinguish the window.
#     """
#     # Sending behavior
#     sent = df_trx[
#         (df_trx["SENDER_ACCOUNT_ID"] == account_id) &
#         (df_trx["date"] >= window_start) &
#         (df_trx["date"] < window_end)
#     ]
    
#     # Receiving behavior
#     received = df_trx[
#         (df_trx["RECEIVER_ACCOUNT_ID"] == account_id) &
#         (df_trx["date"] >= window_start) &
#         (df_trx["date"] < window_end)
#     ]
    
#     return {
#         f"{prefix}_trx_sent": len(sent),
#         f"{prefix}_trx_received": len(received),
#         f"{prefix}_amount_sent": sent["TX_AMOUNT"].sum() if len(sent) > 0 else 0,
#         f"{prefix}_amount_received": received["TX_AMOUNT"].sum() if len(received) > 0 else 0,
#         f"{prefix}_avg_amount_sent": sent["TX_AMOUNT"].mean() if len(sent) > 0 else 0,
#         f"{prefix}_avg_amount_received": received["TX_AMOUNT"].mean() if len(received) > 0 else 0,
#     }

# # Test on one account, one snapshot
# test_account = df_accounts["ACCOUNT_ID"].iloc[5]
# test_snapshot = snapshot_dates[0]
# test_recent_start = test_snapshot - pd.DateOffset(months=1)

# result = compute_intensity_features(df_transactions, test_account, test_recent_start, test_snapshot, "recent")
# print(f"Test account: {test_account}")
# print(f"Snapshot: {test_snapshot.date()}")
# print(f"Window: {test_recent_start.date()} to {test_snapshot.date()}")
# print()
# for k, v in result.items():
#     print(f"  {k}: {v}")


Test account: 5
Snapshot: 2020-04-01
Window: 2020-03-01 to 2020-04-01

  recent_trx_sent: 2
  recent_trx_received: 2
  recent_amount_sent: 140.48
  recent_amount_received: 16.5
  recent_avg_amount_sent: 70.24
  recent_avg_amount_received: 8.25


In [51]:
### This function is more optimized, it filters the data by window first, then filters by account
### Old one -  Loop through each account, and each time filter the ENTIRE 1.3M row dataframe:
### New One: Filter 1,300,000 rows by date → 200,000 rows


def compute_intensity_features(df_trx, snapshot, window_start, prefix):
    """
    Compute behavioral intensity features for all accounts within a time window - 
    prefix: last 1 month or 2 months before last month
    Returns a df with one row per account
    """

    # Filter trx per window

    window_trx = df_trx[(df_trx["date"] >= window_start) & (df_trx["date"] < snapshot)]

    # Sending behavior per client

    sent = (
        df_trx.groupby("SENDER_ACCOUNT_ID")
        .agg(
            trx_sent=("TX_ID", "nunique"),
            amt_sent=("TX_AMOUNT", "sum"),
            avg_amt_sent=("TX_AMOUNT", "mean"),
        )
        .rename_axis("ACCOUNT_ID")
    )

    # Receiving behavior per client

    received = (
        df_trx.groupby("RECEIVER_ACCOUNT_ID")
        .agg(
            trx_received=("TX_ID", "nunique"),
            amt_received=("TX_AMOUNT", "sum"),
            avg_amt_received=("TX_AMOUNT", "mean"),
        )
        .rename_axis("ACCOUNT_ID")
    )

    # Combine features

    features = sent.join(received, "ACCOUNT_ID", "outer").fillna(0)
    # #rename cols
    features.columns =[f"{prefix}_{col}" for col in features.columns]
    return features
